# 🧪 MLflow ResponsesAgent — Tracking, Tracing & Tool Calling

This notebook demonstrates:
- **MLflow ResponsesAgent** for logging, tracking & tracing
- **Structured Request/Response** handling
- **Tool calling** with Python math libraries
- **Chat history** management
- **Token usage** tracking
- **OpenAI API** via LangChain
- **LangGraph** agent loop


**Architecture**
- `LangGraph` compiles a cyclic agent graph: `agent → tools → agent` loop until no more tool calls
- `ToolNode` from LangGraph handles dispatch to all registered tools automatically
- `ChatOpenAI` is bound with tools via `llm.bind_tools(TOOLS)`

**MLflow Integration**
- `mlflow.langchain.autolog()` captures LangChain traces automatically
- A persistent `mlflow.start_run()` spans the whole session
- Each turn gets its own `mlflow.start_span()` with `SpanType.AGENT`
- Per-turn metrics logged: `prompt_tokens`, `completion_tokens`, `latency_ms`, `tool_calls_count`
- Each turn's full structured response is saved as a JSON artifact under `turns/`

**Tools**
| Tool | Library | Purpose |
|------|---------|---------|
| `solve_math` | `sympy` + `numexpr` fallback | Accurate math — e.g. `5 * 3.15 = 15.75` |
| `unit_converter` | Pure Python | km↔miles, kg↔lbs, °C↔°F, m↔ft |
| `get_current_datetime` | stdlib | Current timestamp |

**Structured Models** — Pydantic `ChatRequest`, `ChatResponse`, `TokenUsage`, `ToolCallRecord` wrap every turn cleanly.

**To run:** set `OPENAI_API_KEY`, run all cells, then `mlflow ui` to view traces at `http://127.0.0.1:5000`.


## 1. Install Dependencies

In [1]:
!pip install -q \
    mlflow \
    langchain \
    langchain-openai \
    langchain-core \
    langgraph \
    openai \
    sympy \
    numexpr \
    pydantic

zsh:1: command not found: pip


## 2. Imports & Configuration

In [2]:
import os
import math
import json
import time
import getpass
from typing import Annotated, TypedDict, Literal
from datetime import datetime

import sympy
import numexpr
import mlflow
import mlflow.langchain
from mlflow.entities import SpanType

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
)
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("✅ Imports successful")

ModuleNotFoundError: No module named 'numexpr'

In [ ]:
# ── API Key Setup ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

MODEL_NAME = "gpt-4o-mini"   # swap to gpt-4o for higher quality
MLFLOW_EXPERIMENT = "ResponsesAgent-Demo"
MLFLOW_RUN_NAME   = f"chatbot-session-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"Model  : {MODEL_NAME}")
print(f"Run    : {MLFLOW_RUN_NAME}")

## 3. MLflow Experiment Setup

In [ ]:
# Enable automatic LangChain tracing
mlflow.langchain.autolog()

# Create / activate experiment
mlflow.set_experiment(MLFLOW_EXPERIMENT)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
print(f"Experiment ID   : {experiment.experiment_id}")
print(f"Tracking URI    : {mlflow.get_tracking_uri()}")
print("MLflow UI       : run `mlflow ui` in your terminal to view traces")

## 4. Structured Request / Response Models

In [ ]:
class ChatRequest(BaseModel):
    """Structured input to the chatbot."""
    user_message: str = Field(..., description="The user's message")
    session_id:   str = Field(default="default", description="Conversation session ID")
    max_tokens:   int = Field(default=512, description="Max response tokens")


class ToolCallRecord(BaseModel):
    """Records a single tool invocation."""
    tool_name:   str
    tool_input:  dict
    tool_output: str


class TokenUsage(BaseModel):
    """Token usage for one turn."""
    prompt_tokens:     int = 0
    completion_tokens: int = 0
    total_tokens:      int = 0


class ChatResponse(BaseModel):
    """Structured output from the chatbot."""
    session_id:    str
    user_message:  str
    assistant_reply: str
    tool_calls:    list[ToolCallRecord] = Field(default_factory=list)
    token_usage:   TokenUsage = Field(default_factory=TokenUsage)
    latency_ms:    float = 0.0
    turn_number:   int = 1


print("✅ Pydantic models defined")

## 5. Tool Definitions (Math + General)

In [ ]:
@tool
def solve_math(expression: str) -> str:
    """
    Evaluate a mathematical expression accurately.
    Supports arithmetic, trigonometry, logarithms, algebra, etc.
    Examples: '5 * 3.15', 'sqrt(144)', 'sin(pi/2)', '2**10'
    """
    try:
        # Primary: sympy for symbolic / exact evaluation
        result = sympy.sympify(expression)
        evaluated = float(result.evalf()) if result.is_number else str(result)
        # Return integer when appropriate
        if isinstance(evaluated, float) and evaluated.is_integer():
            return str(int(evaluated))
        return str(evaluated)
    except Exception:
        pass
    try:
        # Fallback: numexpr for numeric-only expressions
        result = numexpr.evaluate(expression)
        return str(float(result))
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


@tool
def get_current_datetime(timezone: str = "UTC") -> str:
    """Return the current date and time."""
    return datetime.now().strftime(f"%Y-%m-%d %H:%M:%S (local machine time)")


@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """
    Convert between common units.
    Supported: km<->miles, kg<->lbs, celsius<->fahrenheit, meters<->feet
    """
    conversions = {
        ("km", "miles"):        lambda v: v * 0.621371,
        ("miles", "km"):        lambda v: v * 1.60934,
        ("kg", "lbs"):          lambda v: v * 2.20462,
        ("lbs", "kg"):          lambda v: v / 2.20462,
        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
        ("meters", "feet"):     lambda v: v * 3.28084,
        ("feet", "meters"):     lambda v: v / 3.28084,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {round(result, 6)} {to_unit}"
    return f"Conversion from '{from_unit}' to '{to_unit}' not supported."


TOOLS = [solve_math, get_current_datetime, unit_converter]
print(f"✅ {len(TOOLS)} tools registered: {[t.name for t in TOOLS]}")

## 6. LangGraph Agent Definition

In [ ]:
# ── Graph State ────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# ── LLM bound with tools ───────────────────────────────────────────────────────
llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=512,
)
llm_with_tools = llm.bind_tools(TOOLS)


# ── Nodes ──────────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = SystemMessage(content=(
    "You are a helpful assistant with access to tools. "
    "For ANY mathematical expression or calculation, ALWAYS use the solve_math tool — "
    "never compute math mentally. Be concise and friendly."
))


def agent_node(state: AgentState) -> AgentState:
    """Call the LLM (with tools bound)."""
    messages = [SYSTEM_PROMPT] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """Route: if the last AI message has tool calls, go to tools; else end."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return "end"


tool_node = ToolNode(TOOLS)


# ── Build Graph ────────────────────────────────────────────────────────────────
graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
graph_builder.add_edge("tools", "agent")

agent_graph = graph_builder.compile()
print("✅ LangGraph agent compiled")

## 7. Visualise the Graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(agent_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualisation unavailable ({e}). Install 'pygraphviz' or 'pyppeteer'.")
    print(agent_graph.get_graph().draw_mermaid())

## 8. Chat Session Manager with MLflow Tracking

In [ ]:
class ChatSession:
    """
    Manages conversation history, MLflow run lifecycle,
    token tracking, and structured request/response logging.
    """

    def __init__(self, session_id: str = "default"):
        self.session_id   = session_id
        self.history: list[BaseMessage] = []
        self.turn         = 0
        self.cumulative   = TokenUsage()
        self.responses: list[ChatResponse] = []

        # Start persistent MLflow run for this session
        self._run = mlflow.start_run(run_name=MLFLOW_RUN_NAME)
        mlflow.log_params({
            "session_id": session_id,
            "model":      MODEL_NAME,
            "tools":      ",".join(t.name for t in TOOLS),
        })
        print(f"🚀 Session '{session_id}' started | MLflow run: {self._run.info.run_id[:8]}...")

    # ── Core method ────────────────────────────────────────────────────────────
    def chat(self, user_message: str) -> ChatResponse:
        request = ChatRequest(
            user_message=user_message,
            session_id=self.session_id,
        )
        self.turn += 1
        t0 = time.time()

        # Build input messages (history + new user message)
        input_messages = self.history + [HumanMessage(content=user_message)]

        # ── MLflow span wraps the entire agent invocation ─────────────────────
        with mlflow.start_span(
            name=f"turn_{self.turn}",
            span_type=SpanType.AGENT,
            attributes={"session_id": self.session_id, "turn": self.turn},
        ) as span:
            span.set_inputs({"user_message": user_message})

            # ── LangGraph invocation ──────────────────────────────────────────
            result = agent_graph.invoke({"messages": input_messages})

            latency_ms   = (time.time() - t0) * 1000
            all_messages = result["messages"]

            # ── Extract final AI reply ────────────────────────────────────────
            ai_messages = [m for m in all_messages if isinstance(m, AIMessage)]
            final_reply = ai_messages[-1].content if ai_messages else "(no response)"

            # ── Collect tool call records ─────────────────────────────────────
            tool_records: list[ToolCallRecord] = []
            ai_with_calls = [m for m in all_messages
                             if isinstance(m, AIMessage) and m.tool_calls]
            tool_results  = {m.tool_call_id: m.content
                             for m in all_messages if isinstance(m, ToolMessage)}

            for ai_msg in ai_with_calls:
                for tc in ai_msg.tool_calls:
                    tool_records.append(ToolCallRecord(
                        tool_name=tc["name"],
                        tool_input=tc["args"],
                        tool_output=tool_results.get(tc["id"], ""),
                    ))

            # ── Token usage (last AI message metadata) ────────────────────────
            usage = TokenUsage()
            last_ai = ai_messages[-1] if ai_messages else None
            if last_ai and hasattr(last_ai, "response_metadata"):
                raw = last_ai.response_metadata.get("token_usage", {})
                usage = TokenUsage(
                    prompt_tokens=raw.get("prompt_tokens", 0),
                    completion_tokens=raw.get("completion_tokens", 0),
                    total_tokens=raw.get("total_tokens", 0),
                )

            # ── Build structured response ─────────────────────────────────────
            response = ChatResponse(
                session_id=self.session_id,
                user_message=user_message,
                assistant_reply=final_reply,
                tool_calls=tool_records,
                token_usage=usage,
                latency_ms=round(latency_ms, 2),
                turn_number=self.turn,
            )

            span.set_outputs({"reply": final_reply, "tool_calls": len(tool_records)})

        # ── Update cumulative token counts ────────────────────────────────────
        self.cumulative.prompt_tokens     += usage.prompt_tokens
        self.cumulative.completion_tokens += usage.completion_tokens
        self.cumulative.total_tokens      += usage.total_tokens

        # ── Log per-turn metrics to MLflow ────────────────────────────────────
        mlflow.log_metrics({
            "prompt_tokens":            usage.prompt_tokens,
            "completion_tokens":        usage.completion_tokens,
            "total_tokens":             usage.total_tokens,
            "latency_ms":               response.latency_ms,
            "tool_calls_count":         len(tool_records),
            "cumulative_total_tokens":  self.cumulative.total_tokens,
        }, step=self.turn)

        # ── Log structured JSON artifact ──────────────────────────────────────
        artifact_path = f"/tmp/turn_{self.turn:03d}.json"
        with open(artifact_path, "w") as f:
            json.dump(response.model_dump(), f, indent=2)
        mlflow.log_artifact(artifact_path, artifact_path="turns")

        # ── Update conversation history (only new messages after last history) ─
        new_messages = all_messages[len(self.history):]
        self.history.extend(new_messages)

        self.responses.append(response)
        return response

    # ── Pretty printer ─────────────────────────────────────────────────────────
    def display_response(self, r: ChatResponse):
        print(f"\n{'='*60}")
        print(f"Turn {r.turn_number} | Session: {r.session_id}")
        print(f"{'='*60}")
        print(f"👤 User    : {r.user_message}")
        if r.tool_calls:
            for tc in r.tool_calls:
                print(f"🔧 Tool    : {tc.tool_name}({tc.tool_input}) → {tc.tool_output}")
        print(f"🤖 Assistant: {r.assistant_reply}")
        print(f"⏱  Latency : {r.latency_ms} ms")
        print(f"🪙 Tokens  : prompt={r.token_usage.prompt_tokens}, "
              f"completion={r.token_usage.completion_tokens}, "
              f"total={r.token_usage.total_tokens}")

    # ── Session teardown ───────────────────────────────────────────────────────
    def end_session(self):
        mlflow.log_metrics({
            "session_total_turns":             self.turn,
            "session_cumulative_total_tokens": self.cumulative.total_tokens,
        })
        mlflow.end_run()
        print(f"\n✅ Session ended. Total turns: {self.turn} | "
              f"Total tokens: {self.cumulative.total_tokens}")


print("✅ ChatSession class ready")

## 9. Run the Chatbot Demo

In [ ]:
# Initialise a new session
session = ChatSession(session_id="demo-session-01")

### Turn 1 — General question (no tool call)

In [ ]:
r1 = session.chat("Hi! What can you help me with?")
session.display_response(r1)

### Turn 2 — Math question (tool call: solve_math)

In [ ]:
r2 = session.chat("What is 5 * 3.15?")
session.display_response(r2)

### Turn 3 — More complex math

In [ ]:
r3 = session.chat("Now calculate sqrt(144) + log(1000) and tell me the result.")
session.display_response(r3)

### Turn 4 — Unit conversion tool

In [ ]:
r4 = session.chat("Convert 100 km to miles.")
session.display_response(r4)

### Turn 5 — Context-aware follow-up (history test)

In [ ]:
r5 = session.chat("Going back to my first math question — multiply that result by 2.")
session.display_response(r5)

### Turn 6 — Current datetime tool

In [ ]:
r6 = session.chat("What is the current date and time?")
session.display_response(r6)

## 10. Session Summary & Token Dashboard

In [ ]:
import pandas as pd

rows = []
for r in session.responses:
    rows.append({
        "Turn":        r.turn_number,
        "User Message": r.user_message[:50] + ("..." if len(r.user_message) > 50 else ""),
        "Tool Calls":  ", ".join(tc.tool_name for tc in r.tool_calls) or "—",
        "Prompt Tok":  r.token_usage.prompt_tokens,
        "Completion Tok": r.token_usage.completion_tokens,
        "Total Tok":   r.token_usage.total_tokens,
        "Latency (ms)": r.latency_ms,
    })

df = pd.DataFrame(rows)
print("\n📊 Session Token & Latency Summary")
print(df.to_string(index=False))
print(f"\n{'─'*60}")
print(f"Cumulative totals  → Prompt: {session.cumulative.prompt_tokens} | "
      f"Completion: {session.cumulative.completion_tokens} | "
      f"Total: {session.cumulative.total_tokens}")

## 11. Chat History Inspection

In [ ]:
print(f"\n💬 Full conversation history ({len(session.history)} messages)\n")
for i, msg in enumerate(session.history):
    role   = type(msg).__name__.replace("Message", "")
    prefix = {"Human": "👤", "AI": "🤖", "Tool": "🔧"}.get(role, "❓")
    content = str(msg.content)[:120]
    print(f"[{i:02d}] {prefix} {role:6s}: {content}")

## 12. Close MLflow Run

In [ ]:
session.end_session()

print("\n🔍 To view runs & traces:")
print("   mlflow ui")
print("   then open http://127.0.0.1:5000 in your browser")

## 13. Interactive Chatbot (Optional Loop)

Run this cell for a live, multi-turn conversation — type `quit` to exit.

In [ ]:
RUN_INTERACTIVE = False   # Set to True to enable interactive mode

if RUN_INTERACTIVE:
    live = ChatSession(session_id="interactive-session")
    print("\n🤖 Chatbot ready! Type 'quit' to exit.\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input or user_input.lower() in {"quit", "exit", "bye"}:
            live.end_session()
            break
        resp = live.chat(user_input)
        live.display_response(resp)